In [ ]:
import pandas as pd

df = pd.read_excel("data_problems.xlsx")
texts = df["Задача"].tolist()
labels = df["Тема"].tolist()

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
labels_encoded = label_encoder.fit_transform(labels)
num_classes = len(label_encoder.classes_)
print("Классы:", label_encoder.classes_)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import torch

model_name = "cointegrated/rubert-tiny2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_classes
)

In [ ]:
from torch.utils.data import Dataset

class MathProblemsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=max_length,
            return_tensors="pt"
        )
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

dataset = MathProblemsDataset(texts, labels_encoded, tokenizer)

In [ ]:
from sklearn.model_selection import train_test_split

train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels_encoded, test_size=0.1, random_state=42, stratify=labels_encoded
)

train_dataset = MathProblemsDataset(train_texts, train_labels, tokenizer)
val_dataset = MathProblemsDataset(val_texts, val_labels, tokenizer)

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    logging_dir='./logs',
    evaluate_during_training=True,  # старый способ
    eval_steps=500,                 # оценка каждые 500 шагов
    save_steps=500,                 # сохранение каждые 500 шагов
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    save_total_limit=2,
    seed=42,
)
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = predictions.argmax(axis=-1)
    from sklearn.metrics import accuracy_score
    return {"accuracy": accuracy_score(labels, predictions)}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

In [ ]:
from simpletransformers.classification import ClassificationModel
import pandas as pd

# Загрузка данных
df = pd.read_excel("data_problems.xlsx")
df = df.rename(columns={"Задача": "text", "Тема": "labels"})

# Преобразование меток в числа
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
df["labels"] = label_encoder.fit_transform(df["labels"])

# Создание модели
model = ClassificationModel(
    "bert", "cointegrated/rubert-tiny2", num_labels=len(label_encoder.classes_),
    use_cuda=False,
    args={
        "reprocess_input_data": True,
        "overwrite_output_dir": True,
        "num_train_epochs": 3,
        "max_seq_length": 512,
        "train_batch_size": 16,
        "eval_batch_size": 16,
        "evaluate_during_training": True,
        "evaluate_during_training_verbose": True,
        "use_cuda": False,  # ВАЖНО: отключаем CUDA

    }
)

# Обучение
model.train_model(df)

# Сохранение
model.save_model("./math-problem-classifier")

In [ ]:
from simpletransformers.classification import ClassificationModel
import pandas as pd
from sklearn.model_selection import train_test_split

# Загрузка данных
df = pd.read_excel("data_problems.xlsx")
df = df.rename(columns={"Задача": "text", "Тема": "labels"})

# Преобразование меток в числа
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
df["labels"] = label_encoder.fit_transform(df["labels"])

# Разделение на train и eval
train_df, eval_df = train_test_split(df, test_size=0.1, random_state=42, stratify=df["labels"])

# Создание модели
model = ClassificationModel(
    "bert", "cointegrated/rubert-tiny2", num_labels=len(label_encoder.classes_),
    use_cuda=False,
        args={
        "reprocess_input_data": True,
        "overwrite_output_dir": True,
        "num_train_epochs": 3,
        "max_seq_length": 512,
        "train_batch_size": 16,
        "eval_batch_size": 16,
        "evaluate_during_training": True,
        "evaluate_during_training_verbose": True,
        "use_cuda": False,
    }
)

# Обучение с валидацией
model.train_model(train_df, eval_df=eval_df)

# Сохранение
model.save_model("./math-problem-classifier")

In [ ]:
from simpletransformers.classification import ClassificationModel
from transformers import AutoModelForSequenceClassification
import pandas as pd
from sklearn.model_selection import train_test_split

# Загрузка данных
df = pd.read_excel("data_problems.xlsx")
df = df.rename(columns={"Задача": "text", "Тема": "labels"})

from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
df["labels"] = label_encoder.fit_transform(df["labels"])

train_df, eval_df = train_test_split(df, test_size=0.1, random_state=42, stratify=df["labels"])

# Загружаем модель
base_model = AutoModelForSequenceClassification.from_pretrained(
    "cointegrated/rubert-tiny2",
    num_labels=len(label_encoder.classes_)
)

# Заморозка backbone
for param in base_model.bert.parameters():
    param.requires_grad = False

# Передаём в simpletransformers
model = ClassificationModel(
    "custom",
    base_model,
    use_cuda=False,
    args={
        "reprocess_input_data": True,
        "overwrite_output_dir": True,
        "num_train_epochs": 3,
        "max_seq_length": 512,
        "train_batch_size": 16,
        "eval_batch_size": 16,
        "evaluate_during_training": True,
        "evaluate_during_training_verbose": True,
        "use_cuda": False,
    }
)

# Обучение
model.train_model(train_df, eval_df=eval_df)

# Сохранение
model.save_model("./math-problem-classifier")